# Insurance Eligibility and Preventable Denial Analysis

# 1. Data Audit

### Objective

The purpose of this section is to understand the structure and quality of the raw encounter data before cleaning or analysis.

The audit will examine:

- dataset dimensions and column names
- data types
- missing values
- duplicate encounter records
- inconsistent categorical values
- potential business-rule conflicts

  healthcare_denial_project/
│
├── notebooks/
│   ├── 01_Data_Audit.ipynb
│   ├── 02_Data_Cleaning.ipynb
│   ├── 03_EDA.ipynb
│
├── data/
│   ├── raw/
│   ├── cleaned/
│
├── dashboard/
│
├── sql/
│
└── README.md

1. Data Audit
    1.1 Data types
    1.2 Missing values
    1.3 Duplicates
    1.4 Category consistency

2. Data Cleaning
    2.1 Fix data types
    2.2 Handle missing values
    2.3 Remove duplicates
    2.4 Standardize categories

In [58]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [59]:
df = pd.read_csv("raw_encounters.csv")

## 1.1 Inspect data types and non-null counts

Before cleaning the dataset, I reviewed each column's data type and non-null count.  
This helps identify fields that may have been imported incorrectly, such as dates or monetary values stored as text.

In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2535 entries, 0 to 2534
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Encounter_ID               2535 non-null   object
 1   Encounter_Date             2535 non-null   object
 2   Department                 2535 non-null   object
 3   Visit_Type                 2535 non-null   object
 4   Payer_Group                2535 non-null   object
 5   Front_End_Issue            2535 non-null   object
 6   RTE_Status                 2490 non-null   object
 7   Coverage_Active            2535 non-null   object
 8   COB_Review_Status          2523 non-null   object
 9   Pre_Registration_Complete  2535 non-null   object
 10  Corrected_Before_Billing   2535 non-null   object
 11  Claim_Denied               2535 non-null   object
 12  Denial_Reason              2535 non-null   object
 13  Allowed_Amount             2535 non-null   object
 14  Denied_A

#### Initial observation

The raw dataset contains 2,535 rows and 17 columns.

All columns were initially imported as `object`, indicating that pandas treated every field as text. Most categorical fields can remain as strings, but the following fields require type conversion:

- `Encounter_Date` should be converted to datetime.
- `Allowed_Amount`, `Denied_Amount`, `Copay_Due`, and `Copay_Collected` should be converted to numeric fields.

The audit also identified missing values in:

- `RTE_Status`
- `COB_Review_Status`

These issues will be investigated and addressed during data cleaning.

## 1.2 Check missing values

Missing values can affect filtering, grouping, and downstream reporting.  
I counted missing values in each column and displayed only the fields that contain missing data.

In [61]:
# Count missing values in each column.
missing_values = df.isna().sum()

# Display only columns with at least one missing value.
missing_values[missing_values > 0].sort_values(ascending=False)

RTE_Status           45
COB_Review_Status    12
dtype: int64

### Finding

Only two columns contain missing values:

- **RTE_Status** has 45 missing values.
- **COB_Review_Status** has 12 missing values.

The percentage of missing values is low relative to the total dataset size (2,535 records).

These missing values will be investigated and handled during the Data Cleaning stage.

&nbsp;



## 1.3 Check duplicate records

Duplicate encounter records can inflate encounter counts, denial rates, and financial metrics.

Before removing duplicates, I first identified whether duplicate encounter IDs exist and examined whether they represent true duplicate records or legitimate repeated visits.

In [62]:
# Count duplicated Encounter_ID values.

duplicate_count = df["Encounter_ID"].duplicated().sum()

print(f"Duplicate Encounter_IDs: {duplicate_count}")

Duplicate Encounter_IDs: 35


In [63]:
# Display duplicated encounters for inspection.

duplicate_records = (
    df[df["Encounter_ID"].duplicated(keep=False)]
      .sort_values("Encounter_ID")
)

duplicate_records

,Encounter_ID,Encounter_Date,Department,Visit_Type,Payer_Group,Front_End_Issue,RTE_Status,Coverage_Active,COB_Review_Status,Pre_Registration_Complete,Corrected_Before_Billing,Claim_Denied,Denial_Reason,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
378,ENC00046,2025-11-03,Orthopedics,Imaging,Commercial,No Front-End Issue,Verified,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,1924.26,0.0,0.0,0.0
746,ENC00046,2025-11-03,Orthopedics,Imaging,Commercial,No Front-End Issue,Verified,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,1924.26,0.0,0.0,0.0
1378,ENC00090,2025-12-07,Orthopedics,Follow-up,Medicaid,No Front-End Issue,Verified,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,133.54,0.0,6.62,6.62
991,ENC00090,2025-12-07,Orthopedics,Follow-up,Medicaid,No Front-End Issue,Verified,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,133.54,0.0,6.62,6.62
1217,ENC00145,2025-12-19,Primary Care,Follow-up,Other Government,No Front-End Issue,Not Required,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,120.85,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
680,ENC02350,2026-01-02,OB/GYN,Office Visit,Commercial,Member ID Mismatch,Unverified,Yes,Not Applicable,Yes,No,No,No Denial,220.45,0.0,0.0,0.0
1575,ENC02375,2026-03-04,Primary Care,Office Visit,Medicare,No Front-End Issue,Not Required,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,197.42,0.0,0.0,0.0
1134,ENC02375,2026-03-04,Primary Care,Office Visit,Medicare,No Front-End Issue,Not Required,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,197.42,0.0,0.0,0.0
847,ENC02416,2026-05-23,Orthopedics,Office Visit,Commercial,No Front-End Issue,Verified,Yes,Not Applicable,No,Not Applicable,No,No Denial,187.24,0.0,53.23,53.23


In [64]:
# Count exact duplicate rows.

exact_duplicates = df.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicates}")

Exact duplicate rows: 35


### Finding

The dataset contains **35 duplicated `Encounter_ID` values**.

After reviewing the duplicated records, all duplicated rows were confirmed to be exact duplicates rather than legitimate repeated patient encounters.

These duplicated records will be removed during the data cleaning stage to prevent inflated encounter counts and financial metrics.

## 1.4 Review categorical consistency

Categorical variables should use consistent labels.

Differences in capitalization, spelling, or formatting can split one category into multiple groups and produce inaccurate summary statistics.

This step identifies inconsistent category labels before standardization.

In [65]:
# Review unique values in Department.
df["Department"].value_counts(dropna=False)

Department
Primary Care     862
OB/GYN           535
Cardiology       411
Orthopedics      360
Imaging          350
Primary Care       4
OBGYN              3
Ob/Gyn             2
Obgyn              2
Ortho              2
Orthopedic         2
Orthopedics        2
Name: count, dtype: int64

### Finding

Several categorical variables contain inconsistent labels caused by differences in capitalization, spelling, abbreviations, and formatting.

For example:

- Department includes "Primary Care", "PRIMARY CARE", and "primary care".
- "OB/GYN", "OBGYN", and "OB/Gyn" represent the same department.
- "Orthopedics" and "Ortho" refer to the same specialty.

If these inconsistencies are not standardized, they will be counted as separate categories and lead to inaccurate summary statistics and visualizations.

These issues will be corrected during the Data Cleaning stage.

# 2. Data Cleaning

The audit identified several data quality issues, including incorrect data types, missing values, duplicate records, and inconsistent categorical labels.

This section applies cleaning procedures to prepare the dataset for reliable analysis and reporting.

## 2.1 Fix data types

Several fields were imported as text (`object`) even though they represent dates or numeric values.

These fields are converted to appropriate data types before further cleaning and analysis.

In [66]:
# Convert encounter date from text to datetime.
# df["Encounter_Date"] = pd.to_datetime(df["Encounter_Date"]) 
# return error, it means there are some text which can not be converted to datetime

# Find rows with invalid date format.
invalid_dates = df[
    pd.to_datetime(
        df["Encounter_Date"],
        errors="coerce"  # if can not convert to datetime, do not return error, return NaT
    ).isna()
]

invalid_dates[["Encounter_ID", "Encounter_Date"]]

,Encounter_ID,Encounter_Date
47,ENC01058,2026/01/13
155,ENC00436,"Jul 08, 2025"
189,ENC01869,"Jun 17, 2026"
300,ENC00812,2026/04/12
303,ENC02125,09/01/2025
...,...,...
2283,ENC00437,"Oct 12, 2025"
2336,ENC00961,08/20/2025
2397,ENC01120,07/08/2025
2405,ENC02277,06/30/2026


### Finding

The Encounter_Date column contains multiple date formats rather than a single standardized format.

Examples include:

- YYYY-MM-DD
- YYYY/MM/DD
- MM/DD/YYYY
- Mon DD, YYYY

A standardized datetime conversion is required before any time-based analysis.

In [67]:
# Convert mixed date formats into datetime.
df["Encounter_Date"] = pd.to_datetime(
    df["Encounter_Date"],
    format="mixed"
)

In [68]:
# Verify the converted data type.
print(df["Encounter_Date"].dtype)

# Confirm that no dates became missing during conversion.
print("Missing dates after conversion:", df["Encounter_Date"].isna().sum())

# Preview the standardized date values.
df[["Encounter_ID", "Encounter_Date"]].head()

datetime64[ns]
Missing dates after conversion: 0


,Encounter_ID,Encounter_Date
0,ENC00105,2026-02-15
1,ENC00964,2026-01-28
2,ENC01970,2025-09-16
3,ENC02282,2026-01-22
4,ENC01957,2025-12-01


### Result

`Encounter_Date` was successfully converted from mixed text formats to a standardized datetime field.

All 2,535 date values were retained, with no missing values introduced during conversion. The field is now ready for monthly trend and time-based analysis.

### Convert monetary fields to numeric values

The monetary columns were imported as text because some values contain currency symbols and thousands separators.

Before calculating totals, averages, or financial impact, these fields must be converted to numeric data types.

In [69]:
# List the monetary fields that require numeric conversion.
amount_columns = [
    "Allowed_Amount",
    "Denied_Amount",
    "Copay_Due",
    "Copay_Collected"
]

In [70]:
# amount_columns is a python list
df[amount_columns].head()

,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
0,1165.11,0.0,50.78,50.78
1,128.36,0.0,0.0,0.0
2,214.66,0.0,0.0,0.0
3,163.2,0.0,0.0,0.0
4,254.44,0.0,0.0,0.0


In [71]:
df[amount_columns].tail()

,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
2530,733.68,0.0,0.0,0.0
2531,305.94,0.0,0.0,0.0
2532,175.38,0.0,0.0,0.0
2533,121.18,0.0,0.0,0.0
2534,809.02,0.0,0.0,0.0


In [72]:
df[amount_columns].sample(10)

,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
285,648.64,0.0,0.0,0.0
1922,142.78,$0.00,0.0,0.0
2003,186.24,0.0,0.0,0.0
2068,142.34,0.0,49.95,49.95
1909,147.42,108.91,0.0,0.0
955,123.26,0.0,47.67,47.67
86,230.69,0.0,0.0,0.0
2031,2155.62,2127.34,0.0,0.0
2130,453.87,325.21,1.55,1.55
2400,185.91,0.0,0.0,0.0


In [73]:
df[amount_columns].describe()

,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
count,2535,2535,2535,2535
unique,2435,299,604,479
top,297.89,0.0,0.0,0.0
freq,3,2219,1876,2012


In [74]:
# Remove currency symbols and thousands separators,
for column in amount_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
# then convert the monetary fields to numeric values.
    df[column] = pd.to_numeric(
        df[column],
# if can not convert to number, convert to NaN
        errors="coerce"
    )

In [75]:
# Verify data types and check whether conversion introduced missing values.
print(df[amount_columns].dtypes)

df[amount_columns].isna().sum()

Allowed_Amount     Float64
Denied_Amount      Float64
Copay_Due          Float64
Copay_Collected    Float64
dtype: object


Allowed_Amount     0
Denied_Amount      0
Copay_Due          0
Copay_Collected    0
dtype: int64

In [76]:
# Preview the converted monetary values.
df[amount_columns].head()

,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
0,1165.11,0.0,50.78,50.78
1,128.36,0.0,0.0,0.0
2,214.66,0.0,0.0,0.0
3,163.2,0.0,0.0,0.0
4,254.44,0.0,0.0,0.0


### Result

All four monetary fields were successfully converted from text (`object`) to numeric (`float64`) data types:

- `Allowed_Amount`
- `Denied_Amount`
- `Copay_Due`
- `Copay_Collected`

The conversion did not introduce any missing values. A preview of the converted records confirmed that the monetary values were preserved correctly and are ready for financial calculations and downstream analysis.

## 2.2 Handle missing values

Only two columns contain missing values.

Rather than deleting records, I first investigated whether the missing values represent meaningful business situations or true data quality issues.

Appropriate handling strategies were then applied to preserve valid encounter records whenever possible.

In [77]:
# Review rows with missing RTE_Status.
df[df["RTE_Status"].isna()].head(10)

,Encounter_ID,Encounter_Date,Department,Visit_Type,Payer_Group,Front_End_Issue,RTE_Status,Coverage_Active,COB_Review_Status,Pre_Registration_Complete,Corrected_Before_Billing,Claim_Denied,Denial_Reason,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
62,ENC02494,2026-06-18,Primary Care,Office Visit,Commercial,No Front-End Issue,NaN,Yes,Not Applicable,Yes,Not Applicable,Yes,Other Non-Preventable Denial,254.64,237.16,43.93,0.0
95,ENC01558,2026-06-09,Cardiology,Follow-up,Medicaid,Member ID Mismatch,NaN,Yes,Not Applicable,Yes,Yes,Yes,Member Identification Mismatch,143.54,111.92,0.0,0.0
156,ENC01711,2026-04-08,OB/GYN,Follow-up,Commercial,Inactive Coverage,NaN,No,Not Applicable,No,Yes,Yes,Coverage Inactive on Date of Service,184.99,164.35,0.0,0.0
178,ENC01948,2025-07-24,Cardiology,Office Visit,Commercial,No Front-End Issue,NaN,Yes,Completed,Yes,Not Applicable,No,No Denial,264.47,0.0,0.0,0.0
203,ENC01149,2026-01-05,Primary Care,Office Visit,Medicare,No Front-End Issue,NaN,Yes,Completed,Yes,Not Applicable,No,No Denial,202.24,0.0,0.0,0.0
227,ENC00906,2026-04-21,Primary Care,Follow-up,Medicare,Incorrect Primary Insurance,NaN,Yes,Missed,Yes,Yes,Yes,Other Insurance Should Be Primary,91.02,85.55,0.0,0.0
286,ENC01341,2025-12-21,Imaging,Procedure,Commercial,No Front-End Issue,NaN,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,658.67,0.0,56.34,0.0
346,ENC00814,2025-10-29,Cardiology,Follow-up,Medicare,No Front-End Issue,NaN,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,217.48,0.0,0.0,0.0
451,ENC01504,2026-01-12,Primary Care,Follow-up,Commercial,No Front-End Issue,NaN,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,104.55,0.0,52.22,0.0
465,ENC01325,2025-10-02,Imaging,Procedure,Commercial,No Front-End Issue,NaN,Yes,Not Applicable,Yes,Not Applicable,No,No Denial,1041.38,0.0,21.75,21.75


### Interpretation

Reviewing the sample records shows several important patterns:

- Almost all encounters have `Coverage_Active = Yes`, indicating that insurance coverage was active for these patients.
- The missing values appear across multiple `Front_End_Issue` categories rather than a single issue type.
- Missing values are also observed across different departments, visit types, and payer groups.

These observations suggest that the missing `RTE_Status` values are unlikely to be caused solely by inactive insurance coverage.

Possible explanations include:

- The eligibility verification was not documented.
- The RTE check was not performed.
- Certain encounter types or workflows may not require an RTE check.

In [78]:
# Review rows with missing COB_Review_Status.
df[df["COB_Review_Status"].isna()].head(10)

,Encounter_ID,Encounter_Date,Department,Visit_Type,Payer_Group,Front_End_Issue,RTE_Status,Coverage_Active,COB_Review_Status,Pre_Registration_Complete,Corrected_Before_Billing,Claim_Denied,Denial_Reason,Allowed_Amount,Denied_Amount,Copay_Due,Copay_Collected
73,ENC01518,2026-04-03,Imaging,Office Visit,Medicare,No Front-End Issue,Verified,Yes,NaN,Yes,Not Applicable,No,No Denial,120.23,0.0,33.63,33.63
216,ENC01758,2026-06-25,Primary Care,Office Visit,Medicaid,Incorrect Primary Insurance,Verified With Alert,Yes,NaN,Yes,No,No,No Denial,237.02,0.0,0.0,0.0
352,ENC00180,2026-04-15,Primary Care,Office Visit,Other Government,No Front-End Issue,Verified,Yes,NaN,Yes,Not Applicable,No,No Denial,161.93,0.0,0.0,0.0
661,ENC01533,2025-07-10,OB/GYN,Office Visit,Commercial,No Front-End Issue,Verified,Yes,NaN,Yes,Not Applicable,No,No Denial,159.21,0.0,0.0,0.0
1280,ENC01373,2025-07-18,OB/GYN,Office Visit,Commercial,Subscriber Information Mismatch,Unverified,Yes,NaN,No,Yes,No,No Denial,144.16,0.0,0.0,0.0
1489,ENC00994,2025-08-15,Primary Care,Imaging,Medicaid,No Front-End Issue,Verified,Yes,NaN,Yes,Not Applicable,No,No Denial,1695.23,0.0,0.0,0.0
1513,ENC02447,2025-08-25,Cardiology,Office Visit,Commercial,No Front-End Issue,Verified,Yes,NaN,Yes,Not Applicable,No,No Denial,305.77,0.0,0.0,0.0
1789,ENC00797,2025-07-16,Primary Care,Imaging,Commercial,Eligibility Not Verified,Unverified,Yes,NaN,No,Yes,No,No Denial,1794.62,0.0,0.0,0.0
1836,ENC00344,2026-03-30,Orthopedics,Procedure,Commercial,No Front-End Issue,Verified,Yes,NaN,Yes,Not Applicable,No,No Denial,1290.72,0.0,0.0,0.0
1902,ENC01258,2026-02-02,OB/GYN,New Patient,Medicare,No Front-End Issue,Verified,Yes,NaN,Yes,Not Applicable,No,No Denial,344.06,0.0,0.0,0.0


### Interpretation

The missing `COB_Review_Status` values also show several patterns:

- Insurance coverage remains active for all reviewed encounters.
- Missing values occur across multiple payer groups, departments, and visit types.
- Different RTE outcomes are present, including **Verified**, **Not Required**, and **Connectivity Error**.

These observations suggest that the missing values are not associated with a single insurance type or registration scenario.

Possible explanations include:

- COB review was not required.
- COB review was not documented.
- The review step was skipped during registration.


In [79]:
# Replace missing status values with "Unknown".
df["RTE_Status"] = df["RTE_Status"].fillna("Unknown")
df["COB_Review_Status"] = df["COB_Review_Status"].fillna("Unknown")

In [80]:
# Verify that missing values were handled.
df[["RTE_Status", "COB_Review_Status"]].isna().sum()

RTE_Status           0
COB_Review_Status    0
dtype: int64

### Cleaning decision

The missing values in `RTE_Status` and `COB_Review_Status` were standardized as `Unknown` rather than being removed or assigned to an existing business status.

This decision was made because the missing records were distributed across multiple departments, payer groups, visit types, and front-end issue categories. No evidence suggested that the missing values represented a single valid business status.

Using `Unknown` preserves all encounter records while explicitly distinguishing missing documentation from confirmed status values.

## 2.3 Remove duplicate records

The data audit identified 35 exact duplicate rows.

Because these records contained identical information across all columns, they were removed to prevent double-counting during analysis.

In [81]:
# Record the dataset size before removing duplicates.
df.shape

(2535, 17)

In [82]:
# Remove exact duplicate rows.
df = df.drop_duplicates()

In [83]:
# Verify the dataset size after removing duplicates.
df.shape

(2500, 17)

### Cleaning result

All 35 exact duplicate records were successfully removed.

The dataset size decreased from **2,535** to **2,500** rows, while the number of columns remained unchanged.

Removing duplicate records prevents inflated encounter counts and ensures that each encounter is represented only once in subsequent analyses.

## 2.4 Standardize category labels

The data audit identified inconsistent category labels caused by differences in capitalization and spelling.

These values were standardized to ensure that identical categories are grouped together during reporting and visualization.

In [84]:
# Review unique values before standardization.
df["Department"].value_counts(dropna=False)

Department
Primary Care     849
OB/GYN           527
Cardiology       407
Orthopedics      353
Imaging          347
Primary Care       4
OBGYN              3
Ob/Gyn             2
Obgyn              2
Ortho              2
Orthopedic         2
Orthopedics        2
Name: count, dtype: int64

In [85]:
# Remove leading and trailing spaces first.
df["Department"] = df["Department"].str.strip()

# Standardize known department variations.
department_mapping = {
    "Primary Care": "Primary Care",
    "Ob/Gyn": "OB/GYN",
    "Obgyn": "OB/GYN",
    "OBGYN": "OB/GYN",
    "Cardiology": "Cardiology",
    "Orthopedics": "Orthopedics",
    "Orthopedic": "Orthopedics",
    "Ortho": "Orthopedics",
    "Imaging": "Imaging"
}

df["Department"] = df["Department"].replace(department_mapping)

In [86]:
# Verify standardized category labels.
df["Department"].value_counts(dropna=False)

Department
Primary Care    853
OB/GYN          534
Cardiology      407
Orthopedics     359
Imaging         347
Name: count, dtype: int64

In [87]:
# Confirm the number of unique department labels.
df["Department"].nunique()

5

### Cleaning result

Department labels were standardized by removing leading and trailing spaces and mapping spelling, capitalization, and abbreviation variations to five approved department names.

The cleaned field now contains only:

- `Primary Care`
- `OB/GYN`
- `Cardiology`
- `Orthopedics`
- `Imaging`

This prevents the same department from being split across multiple categories during analysis.

## Save cleaned dataset

The cleaned dataset is saved for subsequent analysis and dashboard development.

Keeping a separate cleaned dataset preserves the original raw data and ensures that all downstream analyses use the same standardized version.

In [88]:
# Save the cleaned dataset.
df.to_csv("cleaned_encounters.csv", index=False)

In [89]:
# Confirm the cleaned dataset was saved.
import os

os.path.exists("cleaned_encounters.csv")

True

### Cleaning completed

The cleaned dataset was successfully exported as **cleaned_encounters.csv**.

This standardized dataset will be used for all subsequent exploratory analysis and dashboard development.